# 18c - What sets the 0.5 m floor in the 125 m delivery error? (Colab version of notebook 18)

Notebook 17 showed that with perfect burns the median delivery error from the real Checkpoint (125 m) is still 0.50 m. The suspect is the truth-only unmodelled acceleration (an assumed bias of 1e-6 m/s^2 per axis that the filter does not know about). This runs the same seeds 0-49 from 125 m with that acceleration **removed**, in two cases: normal burn errors, and perfect burns (which leaves only navigation).

**Runtime > Change runtime type > T4 GPU**, then **Runtime > Run all**. Colab will ask once to let the notebook use your Google Drive: approve it. Results are written to `MyDrive/touchdown/accel_sens/` so a disconnect cannot lose them, and the run is resumable (finished landings are skipped if you run it again). About 3 hours. Keep this tab open: a running cell counts as activity, a background job does not.

In [ ]:
# 1. Drive (results persist here), then the code, Blender and the NASA terrain
from google.colab import drive
drive.mount('/content/drive')
import os
OUT = '/content/drive/MyDrive/touchdown/accel_sens'
os.makedirs(OUT, exist_ok=True)

In [ ]:
!git clone -q https://github.com/aryanyaksh-art/touch_down.git /tmp/touch_down || (cd /tmp/touch_down && git pull -q)
!sh /tmp/touch_down/scripts/kaggle_setup.sh 2>&1 | tail -5

In [ ]:
# 2. the 195 MB global model, then the wide 125 m terrain (about 15 s)
!cd /tmp/touch_down && TOUCHDOWN_DATA=/tmp/data python scripts/fetch_data.py --global
!cd /tmp/touch_down && TOUCHDOWN_DATA=/tmp/data python scripts/build_wide_dtm.py --out-dir /tmp/data

In [ ]:
# 3. two flight processes side by side, 50 landings each, no unmodelled acceleration (prints a progress line a minute)
import subprocess
run = 'python /tmp/touch_down/scripts/kaggle_run.py --max-hours 6 --wide --seeds 0 50 --accel-scale 0.0'
cases = {'accel0_burn1': '', 'accel0_burn0': '--burn-error-scale 0.0'}
procs = [subprocess.Popen(f"{run} {extra} --out {OUT}/{name}", shell=True) for name, extra in cases.items()]
for p in procs:
    p.wait()
print('both flight processes finished', [p.returncode for p in procs])

In [ ]:
# 4. a quick look at what finished
!ls {OUT}/accel0_burn1 | wc -l; ls {OUT}/accel0_burn0 | wc -l